In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
import pickle

In [ ]:
# Charger et parser le fichier XML
tree = ET.parse('/home/galadriel/dr_benchmark/raw_data/product6.xml')
root = tree.getroot()

# Initialiser une liste pour stocker les informations
data = []

# Parcourir tous les DrugRegulatoryStatus dans le fichier XML
for drug_status in root.findall('DrugRegulatoryStatusList/DrugRegulatoryStatus'):
    # Extraire l'ATC code (s'il existe)
    # atc_code = drug_status.findtext('ATCCode', default='')
    
    # Extraire les informations de chaque Substance
    for substance_association in drug_status.findall('SubstanceDrugRegulatoryStatusAssociationList/SubstanceDrugRegulatoryStatusAssociation'):
        substance = substance_association.find('Substance')
        
        code = substance.findtext('Code', default='')
        chemical_name = substance.findtext('ChemicalName', default='')
        name = substance.findtext('Name', default='')
        
        # Extraire les OrphaCodes et noms des maladies associés
        for disorder in drug_status.findall('DisorderList/Disorder'):
            orpha_code = disorder.findtext('OrphaCode', default='')
            disorder_name = disorder.findtext('Name', default='')
            
            # Extraire les informations de DrugTradeName (s'il y en a)
            trade_names = []
            for trade_name_association in drug_status.findall('DrugTradeNameDrugRegulatoryStatusAssociationList/DrugTradeNameDrugRegulatoryStatusAssociation'):
                trade_name = trade_name_association.findtext('DrugTradeName', default='')
                if trade_name:
                    trade_names.append(trade_name)

            # Si aucun nom commercial n'est trouvé, mettre une chaîne vide
            trade_names_str = ", ".join(trade_names) if trade_names else ''
            
            # Ajouter les informations dans le tableau
            data.append({
                # 'ATCCode': atc_code,
                'Code': code,
                'ChemicalName': chemical_name,
                'Name': name,
                'DrugTradeName': trade_names_str,
                'OrphaCode': orpha_code,
                'DisorderName': disorder_name
            })

# Convertir les données en DataFrame
df = pd.DataFrame(data)
df['Name'] = df['Name'].str.lower()
df['DisorderName'] = df['DisorderName'].str.lower()

In [ ]:
print(df.shape)
df = df.drop_duplicates()
print(df.shape)

In [ ]:
tar_kg = pd.read_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_edges_march_17_2026.csv", sep=",", dtype={"from": str, "rel": str, "to": str})

In [ ]:
tar_kg["rel_full"] = tar_kg["node1_type"] + " " + tar_kg["rel"] + " " + tar_kg["node2_type"]
tar_kg

In [ ]:
tar_kg_nodes = pd.read_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_nodes_mapping_march_17_2026.csv")
tar_kg_nodes

In [ ]:
# Construire les dicts de mapping nom -> id pour drug et disease
drug_node_data    = tar_kg_nodes[tar_kg_nodes["kind"] == "Compound"].drop_duplicates(subset="unify_id")
disease_node_data = tar_kg_nodes[tar_kg_nodes["kind"] == "Disease"].drop_duplicates(subset="unify_id")

In [ ]:
drug_name_to_id    = dict(zip(drug_node_data["name"].str.lower(),    drug_node_data["unify_id"]))
disease_name_to_id = dict(zip(disease_node_data["name"].str.lower(), disease_node_data["unify_id"]))

In [ ]:
# Mapper les entités Orphanet vers les IDs du KG
df["drug_kg_id"]    = df["Name"].str.lower().map(drug_name_to_id)
df["disease_kg_id"] = df["DisorderName"].str.lower().map(disease_name_to_id)

In [ ]:
filtered_df = df.dropna(subset=["drug_kg_id", "disease_kg_id"])
orpha_pairs = set(zip(filtered_df["drug_kg_id"], filtered_df["disease_kg_id"]))
print(f"Paires Orphanet matchées : {len(orpha_pairs)}")


In [ ]:
# Séparer déjà dans le KG vs manquantes
# On travaille uniquement sur les arêtes "Compound indication Disease"
indication_edges = tar_kg[tar_kg["rel_full"] == "Compound indication Disease"]
kg_indication_pairs = set(zip(indication_edges["from"], indication_edges["to"]))

already_in = orpha_pairs & kg_indication_pairs
missing    = orpha_pairs - kg_indication_pairs
print(f"Déjà dans le KG : {len(already_in)}")
print(f"Manquantes (arête absente, nœuds présents) : {len(missing)}")

In [ ]:
# 5. Construire le KG final
# Supprimer les arêtes indication qui sont dans orpha_pairs
tar_kg_filtered = tar_kg[
    ~((tar_kg["rel_full"] == "Compound indication Disease") &
      (tar_kg.apply(lambda r: (r["from"], r["to"]) in orpha_pairs, axis=1)))
]


In [ ]:
# Construire les nouvelles arêtes orpha_indication (already_in + missing)
new_rows = []
for drug_id, disease_id in orpha_pairs:
    new_rows.append({
        "from":       drug_id,
        "node1_type": "Compound",
        "rel":        "orpha_indication",
        "rel_full":   "Compound orpha_indication Disease",
        "to":         disease_id,
        "node2_type": "Disease"
    })

new_edges = pd.DataFrame(new_rows)
new_edges

In [ ]:
# KG final
tar_kg_final = pd.concat([tar_kg_filtered, new_edges], axis=0, ignore_index=True)

tar_kg_final = tar_kg_final.rename(columns={"rel": "rel_init", "rel_full": "rel"})

# Export
tar_kg_final.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_with_orpha_indication.tsv", index=False, sep="\t")

tar_kg_no_orpha = tar_kg_final[tar_kg_final["rel_init"] != "orpha_indication"]
tar_kg_no_orpha.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_without_orpha_indication.tsv", index=False, sep="\t")

In [ ]:
tar_kg_orpha = tar_kg_final[tar_kg_final["rel_init"] == "orpha_indication"]
tar_kg_orpha.to_csv("/home/galadriel/dr_benchmark/knowledge_graph/TarKG_orpha_indication_only.tsv", index=False, sep="\t")